In [ ]:
import os
import sys
import logging
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from azure.ai.projects import AIProjectClient


# Configure a structured enterprise logger streaming directly to stdout
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - [%(levelname)s] - %(name)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("ReconciliationAgent")
logger.info("Notebook execution context initialized successfully.")

C:\Users\alal0\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


2026-07-21 12:10:49,036 - [INFO] - ReconciliationAgent - Notebook execution context initialized successfully.


In [2]:
class ReconciliationState(TypedDict):
    customer_name: str
    invoice_amount: float
    payment_received: float
    has_po: bool
    has_delivery_note: bool
    remittance_remarks: str
    current_priority_rule: int
    assigned_status: str
    deduction_reason_code: str
    audit_trail: list

logger.info("Centralized state memory bank structurally defined.")

2026-07-21 12:10:49,045 - [INFO] - ReconciliationAgent - Centralized state memory bank structurally defined.


In [3]:
import time
from collections import namedtuple

# Create a mock token object matching the Azure core signature
AccessToken = namedtuple("AccessToken", ["token", "expires_on"])

class StaticTokenCredential:
    """Wrapper to safely route a hardcoded string key into token-based client initializers."""
    def __init__(self, key: str):
        self.key = key

    def get_token(self, *scopes, **kwargs):
        # Return the key as the token payload with a distant expiration timestamp
        return AccessToken(token=self.key, expires_on=int(time.time()) + 3600)

try:
    # Wrap the hardcoded key into a token compatible format
    credential = StaticTokenCredential(AZURE_AI_PROJECT_KEY)
    
    project_client = AIProjectClient(
        endpoint=AZURE_AI_PROJECT_ENDPOINT,
        credential=credential
    )
    logger.info(f"Connected securely to Azure AI Project Endpoint: {AZURE_AI_PROJECT_ENDPOINT}")
except Exception as ex:
    logger.error(f"Critical initialization failure at cloud infrastructure layer: {str(ex)}")
    raise ex

2026-07-21 12:10:49,054 - [INFO] - ReconciliationAgent - Connected securely to Azure AI Project Endpoint: https://hub-accenture-training.services.ai.azure.com/api/projects/project-agentic-matching


In [4]:
def process_priority_1(state: ReconciliationState) -> dict:
    logger.info("--- ENTERING NODE: Priority 1 Matcher ---")
    trail = state.get("audit_trail", [])[:]
    trail.append("Evaluated Priority 1: Customer Name + PO Number + Amount")
    
    logger.info(f"Target Transaction Analysis -> Customer: {state['customer_name']}, Has PO: {state['has_po']}")
    
    if state["has_po"]:
        if state["payment_received"] == state["invoice_amount"]:
            logger.info("Priority 1 Match Successful. Perfect balance alignment verified.")
            return {"assigned_status": "Closed", "current_priority_rule": 1, "audit_trail": trail}
        else:
            logger.info("Priority 1 Match flagged balance discrepancy. Shifting to analysis.")
            return {"assigned_status": "Partial Match", "current_priority_rule": 1, "audit_trail": trail}
    
    logger.warning("Priority 1 Failure: Purchase Order identifier completely missing from record.")
    return {"assigned_status": "PO_MISSING", "current_priority_rule": 1, "audit_trail": trail}

In [5]:
def process_priority_2(state: ReconciliationState) -> dict:
    logger.info("--- ENTERING NODE: Priority 2 Matcher (Fallback) ---")
    trail = state.get("audit_trail", [])[:]
    trail.append("Evaluated Priority 2: Customer Name + Delivery Number + Amount")
    
    logger.info(f"Checking secondary indicators -> Has Delivery Note: {state['has_delivery_note']}")
    
    if state["has_delivery_note"]:
        variance = state["invoice_amount"] - state["payment_received"]
        logger.info(f"Delivery indicator matched. Calculated ledger gap variance: ${variance:.2f}")
        return {"assigned_status": "Partial Match", "current_priority_rule": 2, "audit_trail": trail}
        
    logger.error("Priority 2 Failure: Secondary delivery note record validation failed.")
    return {"assigned_status": "UAC", "current_priority_rule": 2, "audit_trail": trail}

In [6]:
def process_deduction_identification(state: ReconciliationState) -> dict:
    logger.info("--- ENTERING NODE: Deduction Reason Identification Agent (RAG) ---")
    trail = state.get("audit_trail", [])[:]
    trail.append("Evaluated Capstone 2: Deduction Reason Classification via Cloud Inference")
    
    variance = state["invoice_amount"] - state["payment_received"]
    
    if variance <= 5.00:
        logger.info(f"Discrepancy of ${variance:.2f} falls within the auto-write off boundary.")
        return {"assigned_status": "Closed", "deduction_reason_code": "WRITE_OFF", "audit_trail": trail}
        
    logger.info(f"Extracting contexts from raw remittance remark payload: '{state['remittance_remarks']}'")
    
    # Trigger client endpoint runtime targeting your hardcoded deployment identifier
    with project_client.get_openai_client() as openai_client:
        prompt_instruction = f"""
        You are a financial compliance engine. Classify this remittance markdown statement into a reason code:
        Remarks: "{state['remittance_remarks']}"
        Supported Reason Mapping:
        - Pricing Issue -> D01
        - Freight Claim -> D02
        - Damage Claim -> D03
        - Tax Difference -> D04
        - Discount Taken -> D05
        Return ONLY the alphanumeric code (e.g., D01).
        """
        
        
        cloud_response = openai_client.chat.completions.create(
            model=AZURE_AI_MODEL_DEPLOYMENT_NAME,
            messages=[{"role": "user", "content": prompt_instruction}]
        )
        
        extracted_code = cloud_response.choices[0].message.content.strip()
        logger.info(f"Azure Generative Inference result validated. Assigned Reason Code: {extracted_code}")
        
        return {"deduction_reason_code": extracted_code, "audit_trail": trail}

In [7]:
def dynamic_router(state: ReconciliationState) -> str:
    status = state["assigned_status"]
    if status == "PO_MISSING":
        logger.info("Routing decision: Priority 1 failed -> Branching out to Priority 2 Node.")
        return "priority_2"
    elif status == "Partial Match":
        logger.info("Routing decision: Matching gap detected -> Directing to Deduction RAG Agent.")
        return "deduction_agent"
    else:
        logger.info("Routing decision: Termination pattern encountered. Closing graph execution loop.")
        return END

# Construct and assemble the state machine blueprint canvas
workflow_builder = StateGraph(ReconciliationState)

workflow_builder.add_node("priority_1", process_priority_1)
workflow_builder.add_node("priority_2", process_priority_2)
workflow_builder.add_node("deduction_agent", process_deduction_identification)

workflow_builder.add_edge(START, "priority_1")
workflow_builder.add_conditional_edges("priority_1", dynamic_router)
workflow_builder.add_conditional_edges("priority_2", dynamic_router)
workflow_builder.add_edge("deduction_agent", END)

reconciliation_agent_app = workflow_builder.compile()
logger.info("LangGraph workflow orchestration successfully compiled and validated.")

2026-07-21 12:10:49,104 - [INFO] - ReconciliationAgent - LangGraph workflow orchestration successfully compiled and validated.


In [8]:
test_payload: ReconciliationState = {
    "customer_name": "Accenture Enterprise Vendor",
    "invoice_amount": 10000.00,
    "payment_received": 9500.00,
    "has_po": False,
    "has_delivery_note": True,
    "remittance_remarks": "Pricing variance encountered on purchase order item list, short paid 500.",
    "current_priority_rule": 0,
    "assigned_status": "Open",
    "deduction_reason_code": "NONE",
    "audit_trail": []
}

logger.info("Launching execution run with short-payment sample context...")
final_output_state = reconciliation_agent_app.invoke(test_payload)

print("\n==== RUN TRANSACTION EXECUTION COMPLETED SIMULATION SUMMARY ====")
print(f"Final Reconciliation End Status : {final_output_state['assigned_status']}")
print(f"Executed Priority Fallback Rule : Priority {final_output_state['current_priority_rule']}")
print(f"Extracted Dispute Reason Code   : {final_output_state['deduction_reason_code']}")
print(f"Platform History Audit Log Trace:")
for index, trace in enumerate(final_output_state["audit_trail"], 1):
    print(f"  {index}. {trace}")

2026-07-21 12:10:49,122 - [INFO] - ReconciliationAgent - Launching execution run with short-payment sample context...
2026-07-21 12:10:49,133 - [INFO] - ReconciliationAgent - --- ENTERING NODE: Priority 1 Matcher ---
2026-07-21 12:10:49,134 - [INFO] - ReconciliationAgent - Target Transaction Analysis -> Customer: Accenture Enterprise Vendor, Has PO: False
2026-07-21 12:10:49,136 - [WARNING] - ReconciliationAgent - Priority 1 Failure: Purchase Order identifier completely missing from record.
2026-07-21 12:10:49,137 - [INFO] - ReconciliationAgent - Routing decision: Priority 1 failed -> Branching out to Priority 2 Node.
2026-07-21 12:10:49,139 - [INFO] - ReconciliationAgent - --- ENTERING NODE: Priority 2 Matcher (Fallback) ---
2026-07-21 12:10:49,140 - [INFO] - ReconciliationAgent - Checking secondary indicators -> Has Delivery Note: True
2026-07-21 12:10:49,141 - [INFO] - ReconciliationAgent - Delivery indicator matched. Calculated ledger gap variance: $500.00
2026-07-21 12:10:49,142 -

In [9]:
import ipywidgets as widgets
from IPython.display import display, HTML
import io
import logging

# Set up an internal string stream capture to grab the log statements in real-time
log_capture_string = io.StringIO()
log_handler = logging.StreamHandler(log_capture_string)
log_handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))

# Attach this runtime catcher to our existing agent logger
agent_logger = logging.getLogger("ReconciliationAgent")
agent_logger.addHandler(log_handler)

# --- Dynamic State Rendering Pipeline ---
def update_dashboard_ui(payload=None, final_state=None, active_logs=""):
    """Generates and injects dynamic HTML based on active runtime states."""
    
    # Setup baseline defaults if no execution has happened yet
    cust_name = payload["customer_name"] if payload else "No Active Session"
    inv_amt = payload["invoice_amount"] if payload else 0.0
    pmt_rec = payload["payment_received"] if payload else 0.0
    variance = inv_amt - pmt_rec
    
    status = final_state["assigned_status"] if final_state else "Open"
    reason_code = final_state["deduction_reason_code"] if final_state else "PENDING"
    remarks = payload["remittance_remarks"] if payload else "Awaiting execution payload ingestion..."
    
    # Process the text log trace structure
    log_display = active_logs if active_logs else "[SYSTEM] Ready. Configure inputs below and click 'Execute System Run'."
    log_html_lines = log_display.replace("\n", "<br>")
    
    html_layout = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background-color: #f3f4f6; padding: 20px; border-radius: 8px;">
        <div style="max-width: 900px; margin: 0 auto; background: #ffffff; padding: 25px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">
            <div style="border-bottom: 2px solid #e5e7eb; padding-bottom: 15px; margin-bottom: 20px;">
                <h2 style="margin: 0; color: #111827;">Agentic Reconciliation Control Panel App</h2>
            </div>

            <!-- Live Dashboard Metrics -->
            <div style="display: flex; gap: 20px; margin-bottom: 20px;">
                <div style="flex: 1; padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa;">
                    <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Customer Scope</label>
                    <div style="font-size: 15px; font-weight: 600; color: #1f2937; margin-top: 5px;">{cust_name}</div>
                </div>
                <div style="flex: 1; padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa;">
                    <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Calculated Discrepancy</label>
                    <div style="font-size: 15px; font-weight: 600; color: #1f2937; margin-top: 5px;">${variance:,.2f}</div>
                </div>
                <div style="flex: 1; padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa;">
                    <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Status</label>
                    <div style="margin-top: 5px;"><span style="display: inline-block; padding: 4px 10px; font-size: 12px; font-weight: bold; border-radius: 12px; background-color: #fef3c7; color: #d97706;">{status}</span></div>
                </div>
                <div style="flex: 1; padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa;">
                    <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Dispute Code (RAG)</label>
                    <div style="margin-top: 5px;"><span style="display: inline-block; padding: 4px 10px; font-size: 12px; font-weight: bold; border-radius: 12px; background-color: #dbeafe; color: #2563eb;">{reason_code}</span></div>
                </div>
            </div>

            <!-- Grounding Remittance Text Display -->
            <div style="padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa; margin-bottom: 20px;">
                <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Ingested Remittance Context</label>
                <p style="margin: 6px 0 0 0; font-style: italic; color: #374151; font-size: 13px;">"{remarks}"</p>
            </div>

            <!-- Real-Time Trace Logging Panel -->
            <label style="font-size: 11px; color: #6b7280; font-weight: bold; text-transform: uppercase;">LangGraph Engine Reasoning Execution Logs</label>
            <div style="background: #1f2937; color: #f9fafb; font-family: monospace; padding: 15px; border-radius: 6px; height: 160px; overflow-y: auto; margin-top: 8px; font-size: 12px; line-height: 1.4;">
                {log_html_lines}
            </div>
        </div>
    </div>
    """
    ui_output.clear_output(wait=True)
    with ui_output:
        display(HTML(html_layout))

# --- Widget Control Elements Initialization ---
input_customer = widgets.Text(value="Accenture Enterprise Vendor", description="Customer:")
input_invoice = widgets.FloatText(value=10000.00, description="Invoice Amt:")
input_payment = widgets.FloatText(value=9500.00, description="Payment Recv:")
input_has_po = widgets.Checkbox(value=False, description="Has Valid PO Indicator")
input_has_dn = widgets.Checkbox(value=True, description="Has Delivery Note Backup")
input_remarks = widgets.Textarea(value="Pricing variance encountered on purchase order item list, short paid 500.", description="Remarks:", layout=widgets.Layout(width='90%', height='60px'))
btn_execute = widgets.Button(description="Execute System Run", button_style="success", icon="play", layout=widgets.Layout(width='90%', margin='10px 0px'))

# Layout form container
form_box = widgets.VBox([
    widgets.HTML("<h4>Configure Raw Ledger Transaction Variables</h4>"),
    input_customer, input_invoice, input_payment, input_has_po, input_has_dn, input_remarks, btn_execute
], layout=widgets.Layout(padding="15px", border="1px solid #ccc", background_color="#fafafa", width="350px", border_radius="8px"))

ui_output = widgets.Output()
app_container = widgets.HBox([form_box, ui_output], layout=widgets.Layout(gap="20px"))

# --- Engine Execution Trigger Connection ---
def trigger_agent_execution(b):
    # Reset tracking log capture streams prior to execution trigger
    log_capture_string.truncate(0)
    log_capture_string.seek(0)
    
    # 1. Package dynamic parameters gathered from HTML input fields into graph dictionary form
    payload_state: ReconciliationState = {
        "customer_name": input_customer.value,
        "invoice_amount": float(input_invoice.value),
        "payment_received": float(input_payment.value),
        "has_po": bool(input_has_po.value),
        "has_delivery_note": bool(input_has_dn.value),
        "remittance_remarks": input_remarks.value,
        "current_priority_rule": 0,
        "assigned_status": "Open",
        "deduction_reason_code": "NONE",
        "audit_trail": []
    }
    
    agent_logger.info("Application layer event received. Passing state payload variables into compiled LangGraph engine...")
    
    try:
        # 2. Invoke the compiled LangGraph application instance safely 
        runtime_final_state = reconciliation_agent_app.invoke(payload_state)
        
        # 3. Read the populated trace values out of the captured logs
        captured_logs = log_capture_string.getvalue()
        
        # 4. Refresh the interactive HTML container state fields dynamically
        update_dashboard_ui(payload=payload_state, final_state=runtime_final_state, active_logs=captured_logs)
        
    except Exception as run_err:
        agent_logger.error(f"Execution run halted by runtime exception: {str(run_err)}")
        update_dashboard_ui(payload=payload_state, final_state=None, active_logs=log_capture_string.getvalue())

# Link the button click handle action to the function execution mapping
btn_execute.on_click(trigger_agent_execution)

# Display the main App frame inside the notebook cell context
display(app_container)

# Trigger initialization run to establish screen graphics elements
update_dashboard_ui()